In [15]:
import os
import openai
import logging
from tqdm import tqdm
from langchain_community.document_loaders import PyPDFLoader, UnstructuredPowerPointLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.text_splitter import CharacterTextSplitter
from langchain.vectorstores import FAISS
from langchain.embeddings.openai import OpenAIEmbeddings
from langchain.document_loaders import (
    UnstructuredWordDocumentLoader, UnstructuredExcelLoader,
    UnstructuredCSVLoader, TextLoader, JSONLoader
)
from langchain.schema import Document 


# 初期設定(ログとAPIキーの設定)

In [4]:
import os
import logging
from tqdm import tqdm
from langchain_community.document_loaders import (
    PyPDFLoader, UnstructuredPowerPointLoader, UnstructuredWordDocumentLoader,
    UnstructuredExcelLoader, UnstructuredCSVLoader, TextLoader, JSONLoader
)
import geopandas as gpd

# ログの設定
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# **対応するローダーを定義**
LOADERS = {
    ".pdf": PyPDFLoader,
    ".pptx": UnstructuredPowerPointLoader,
    ".docx": UnstructuredWordDocumentLoader,
    ".doc": UnstructuredWordDocumentLoader,
    ".xlsx": UnstructuredExcelLoader,
    ".xlsm": UnstructuredExcelLoader,
    ".xls": UnstructuredExcelLoader,
    ".csv": UnstructuredCSVLoader,
    ".txt": TextLoader,
    ".json": JSONLoader
}

# **GIS データ対応**
GIS_LOADERS = {".shp", ".geojson"}

# **スキップすべきファイル**
IGNORED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".gif", ".exe", ".zip", ".tar", ".mp3", ".mp4"}

def get_all_files(path):
    """ 指定フォルダ内のすべてのファイルパスを取得 """
    if not os.path.exists(path):
        logging.error(f"❌ 指定されたフォルダが見つかりません: {path}")
        return []

    all_files = []
    for root, _, files in os.walk(path):
        for file in files:
            all_files.append(os.path.join(root, file))
    return all_files

def load_documents_from_folder(path):
    """ 指定フォルダ内のドキュメントをロード """
    success_count = 0
    skip_count = 0
    skip_data = {}  # スキップされた拡張子のカウント用
    success_file_list = []
    documents = []
    
    path_list = get_all_files(path)
    
    if not path_list:
        logging.error("❌ 読み込むファイルがありません。")
        return [], 0, [], {}, 0

    for filepath in tqdm(path_list, desc=f"📖 {len(path_list)} ファイルをスキャン中"):
        try:
            filename = os.path.basename(filepath)
            ext = os.path.splitext(filepath)[1].lower()  # 拡張子を小文字に変換
            
            # **macOSのメタデータファイル・不要な拡張子をスキップ**
            if filename.startswith("._") or ext in IGNORED_EXTENSIONS:
                logging.warning(f"⚠️ スキップ: {filepath}")
                continue

            # **一般的なドキュメントの処理**
            if ext in LOADERS:
                logging.info(f"📄 {ext.upper()} を読み込み中: {filepath}")
                loader = LOADERS[ext](filepath)
                loaded_docs = loader.load()
                documents.extend(loaded_docs)
                success_count += 1
                success_file_list.append(filepath)

            # **GIS データ（GeoJSON, Shapefile）**
            elif ext in GIS_LOADERS:
                logging.info(f"🌍 GISデータ {ext.upper()} を読み込み中: {filepath}")
                gdf = gpd.read_file(filepath)
                text_data = gdf.to_json()
                documents.append({"text": text_data, "metadata": {"source": filepath}})
                success_count += 1
                success_file_list.append(filepath)

            # **XML, HTML の読み込み**
            elif ext in {".xml", ".html"}:
                logging.info(f"📜 {ext.upper()} を読み込み中: {filepath}")
                with open(filepath, "r", encoding="utf-8") as file:
                    text_data = file.read()
                documents.append({"text": text_data, "metadata": {"source": filepath}})
                success_count += 1
                success_file_list.append(filepath)

            # **プログラムコード（Python, Java, C++ など）**
            elif ext in {".py", ".java", ".c", ".cpp", ".js", ".css"}:
                logging.info(f"💻 {ext.upper()} を読み込み中: {filepath}")
                with open(filepath, "r", encoding="utf-8") as file:
                    code = file.read()
                documents.append({"text": code, "metadata": {"source": filepath}})
                success_count += 1
                success_file_list.append(filepath)

            # **未対応の拡張子**
            else:
                logging.warning(f"⚠️ 未対応のファイル形式（スキップ）: {filepath}")
                skip_count += 1
                skip_data[ext] = skip_data.get(ext, 0) + 1  # スキップした回数を記録

        except Exception as e:
            logging.error(f"❌ エラー: {filepath} の読み込みに失敗しました: {e}")

    return documents, success_count, success_file_list, skip_data, skip_count

In [8]:
# ログの設定
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# OpenAI APIキーの取得
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    logging.warning("⚠️ OpenAI APIキーが設定されていません。環境変数 OPENAI_API_KEY を設定してください。")
else:
    openai.api_key = openai_api_key


# データが格納されているフォルダを指定（絶対パスを取得）
PATH = os.path.abspath("/Volumes/NO NAME/中央大学")
logging.info(f"📂 読み込むフォルダ: {PATH}")

# ドキュメントのロード
documents, success_count, success_file_list, skip_data, skip_count = load_documents_from_folder(PATH)

if not documents:
    logging.error("❌ 読み込まれたドキュメントがありません。処理を中止します。")
    exit()

# ロード成功したファイルを一覧表示
logging.info(f"✅ {success_count} 件のファイルを読み込みました。")
logging.info(f"📝 成功したファイルリスト: {success_file_list}")

# スキップした拡張子の一覧を表示
if skip_count > 0:
    logging.warning(f"⚠️ スキップしたファイル数: {skip_count} 個")
    logging.warning(f"🔍 スキップした拡張子一覧: {skip_data}")

document_objects = [
    Document(page_content=doc["text"], metadata=doc["metadata"]) if isinstance(doc, dict) else doc
    for doc in documents
]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, 
    chunk_overlap=100,
    separators=["\n\n", "\n", "。", "、", " "]  # ← 文単位で区切る
)
split_docs = text_splitter.split_documents(document_objects)

# ベクトルデータベースに埋め込み
logging.info("🛠️ ベクトルデータベースを作成中...")
try:
    vector_db = FAISS.from_documents(split_docs, OpenAIEmbeddings())
    vector_db.save_local("faiss_index")
    logging.info("✅ ベクトルデータベースの作成 & 保存完了")
except Exception as e:
    logging.error(f"❌ ベクトルデータベースの作成または保存に失敗しました: {e}")
    exit()

logging.info("🚀 処理完了！")

2025-03-03 20:28:33,026 - INFO - 📂 読み込むフォルダ: /Volumes/NO NAME/中央大学
📖 5510 ファイルをスキャン中:   0%|          | 0/5510 [00:00<?, ?it/s]2025-03-03 20:28:33,674 - WARNING - ⚠️ 未対応のファイル形式（スキップ）: /Volumes/NO NAME/中央大学/.DS_Store
2025-03-03 20:28:33,675 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/._.DS_Store
2025-03-03 20:28:33,675 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/._統計学_all
2025-03-03 20:28:33,675 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/._プログラミング言語及び演習第一
2025-03-03 20:28:33,676 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/._プログラミング言語及演習第二
2025-03-03 20:28:33,676 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/._統計学演習_all
2025-03-03 20:28:33,677 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/._データ解析第二
2025-03-03 20:28:33,677 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/OR第一/スクリーンショット 2024-05-19 10.34.21.png
2025-03-03 20:28:33,678 - WARNING - ⚠️ スキップ: /Volumes/NO NAME/中央大学/OR第一/._スクリーンショット 2024-05-19 10.34.21.png
2025-03-03 20:28:33,678 - WARNING - ⚠️ 未対応のファイル形式（スキップ）: /Volumes/NO NAME/中央大学/OR第

In [10]:
success_file_list

['/Volumes/NO NAME/中央大学/多文化共生論/多文化共生論_課題_悪役視点による再構成.pdf',
 '/Volumes/NO NAME/中央大学/多文化共生論/期末レポート.pdf',
 '/Volumes/NO NAME/中央大学/多文化共生論/多文化共生論 レポート.pdf',
 '/Volumes/NO NAME/中央大学/企業データ分析/オリエンタルランド_有価証券報告書.pdf',
 '/Volumes/NO NAME/中央大学/企業データ分析/企業データ分析.pdf',
 '/Volumes/NO NAME/中央大学/企業データ分析/富士急_有価証券報告書.pdf',
 '/Volumes/NO NAME/中央大学/英語表現演習2/英語表現演習課題_23D7104001I_YutoTakagi.pdf',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/logging.py',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/signals.py',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/sessions.py',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/config.py',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/templating.py',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/globals.py',
 '/Volumes/NO NAME/中央大学/FlaskWebApp/venv/lib/python3.11/site-packages/flask/__init_

In [12]:
from langchain.chains.question_answering import load_qa_chain
from langchain.llms import OpenAI
from langchain_openai import ChatOpenAI

In [ ]:
# FAISS のベクトルデータベースをロード
vector_db = FAISS.load_local("faiss_index", OpenAIEmbeddings(), allow_dangerous_deserialization=True)

# 質問を設定
query = "中央大学のプログラミング言語及び演習第二の授業資料について教えてください"

# 類似度検索を実行
docs = vector_db.similarity_search(query)

# ChatGPTに質問と関連文書を送信
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
chain = load_qa_chain(llm, chain_type="stuff")
response = chain.run(input_documents=docs, question=query)

print("🤖 回答:")
print(response)


2025-03-03 21:07:21,597 - INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
2025-03-03 21:07:23,453 - INFO - HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


🤖 回答:
申し訳ありませんが、その情報については提供することができません。
